In [1]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")
# cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")

boco_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_boco_wind_vec.nc"
clust_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_cluster_wind_vec.nc"

# Dask cluster

client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 7,Total memory: 32.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33749,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34945,Total threads: 1
Dashboard: /proxy/46685/status,Memory: 4.57 GiB
Nanny: tcp://127.0.0.1:46441,


In [2]:
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GULLRWF2',
            'GUNNING1',
            'BANGOWF1',
            'BANGOWF2',
            'COLWF01',
            'WOODLWN1',
            'BOCORWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [1]:
ds = xr.open_dataset(boco_ds, chunks={'time':1}, engine='h5netcdf')
ds

NameError: name 'xr' is not defined

In [4]:
@delayed
def plot_frame(u, v, lat, lon, t, output_dir, quiver_scale, extent, cluster=cluster, highlight_id='BOCORWF1'):
    """
    Plot wind vectors with optional cluster points.
    
    cluster: DataFrame with columns ['ID', 'lat', 'lon']
    highlight_id: specific ID to highlight in red
    """
    speed = np.sqrt(u**2 + v**2)
    plt.figure(figsize=(10,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, lw=1.5)
    ax.add_feature(cfeature.BORDERS, linestyle='-', lw=1.5)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.5)
    
    # Contour of wind speed
    plt.contourf(lon, lat, speed, cmap='viridis', transform=ccrs.PlateCarree())
    plt.colorbar(label='Wind speed (m/s)')
    
    # Quiver vectors
    plt.quiver(lon, lat, u, v, scale=quiver_scale, color='white', transform=ccrs.PlateCarree())
    
    # Plot cluster points
    if cluster is not None:
        # All points in orange
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", alpha=0.6, s=50, label="Wind Farms", zorder=5, transform=ccrs.PlateCarree())
        
        # Highlight one point in red
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'], 
                       color="red", alpha=0.6, s=80, label=f"ID {highlight_id}", zorder=6,
                       transform=ccrs.PlateCarree())

    plt.title(f'Wind vectors at {str(t)}')
    
    dt_str = np.datetime_as_string(t, unit='m')
    dt_str = dt_str.replace('-', '')[2:8] + '_' + dt_str[11:13] + dt_str[14:16]
    filename = os.path.join(output_dir, f'wind_{dt_str}.png')
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    return filename


# -------------------------------
# Function to save all frames for a given hour or all times
# -------------------------------
def save_wind_frames(ds, u_var='ua100m', v_var='va100m', hour=None, output_dir='wind_plots', step=5, quiver_scale=150, parallel=True, extent=None):
    """
    Save plots of all entries at a specific hour (or all time steps if hour=None).
    """
    # Select times
    if hour is not None:
        ds_sub = ds.sel(time=ds.time.dt.hour == hour)
    else:
        ds_sub = ds
    
    tasks = []
    for i in range(len(ds_sub.time)):
        u = ds_sub[u_var].isel(time=i)[::step, ::step]
        v = ds_sub[v_var].isel(time=i)[::step, ::step]
        lat = ds_sub['lat'][::step].values
        lon = ds_sub['lon'][::step].values
        t = ds_sub.time[i].values
        
        if parallel:
            tasks.append(plot_frame(u, v, lat, lon, t, output_dir, quiver_scale, extent))
        else:
            # For serial execution, just call the delayed function and compute immediately
            plot_frame(u, v, lat, lon, t, output_dir, quiver_scale, extent).compute()
    
    if parallel:
        results = compute(*tasks)
        return results
    return None

In [ ]:
# Local Flows
save_wind_frames(ds,
                 hour=17,
                 output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17_zoom',
                 step=2,
                 quiver_scale=500,
                 parallel=True,
                 extent=[147.5, 151, -38.5, -33.5])

In [6]:
# Local Flows
save_wind_frames(ds,
                 hour=17,
                 output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17',
                 step=5,
                 quiver_scale=400,
                 parallel=True,
                 extent=[145, 153, -40, -32.5])

('/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17/wind_151219_1700.png',
 '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17/wind_151220_1700.png',
 '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17/wind_151221_1700.png',
 '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17/wind_151222_1700.png',
 '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17/wind_160111_1700.png',
 '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17/wind_160112_1700.png',
 '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17/wind_160113_1700.png',
 '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17/wind_160114_1700.png',
 '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17/wind_160120_1700.png',
 '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17/wind_160121_1700.png',
 '/g/data/ng72/ms5578/ID_HW_BARRA/data/o

In [7]:
# save_wind_frames(ds,
#                  hour=12,
#                  output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/wind_hour_12',
#                  step=2,
#                  quiver_scale=400,
#                  parallel=True,
#                  extent=[147, 152, -40, -34])

In [8]:
# save_wind_frames(ds,
#                  hour=6,
#                  output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/wind_hour_6',
#                  step=2,
#                  quiver_scale=400,
#                  parallel=True,
#                  extent=[145, 153, -40, -33])